Implementation of Gaussian Process Regression (GPR)

Import Required Libraries

In [ ]:
# ==========================================================
# BLOCK 1: IMPORT LIBRARIES
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    WhiteKernel,
    ConstantKernel
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

Load Dataset

In [ ]:
# ==========================================================
# BLOCK 2: LOAD DATASET
# ==========================================================

df = pd.read_csv(r"C:\Users\Rupa\Downloads\material_modelling_project\main_dataset_superconductor\train.csv")

print("Dataset Shape:", df.shape)

df.head()


BLOCK 3: DATA EXPLORATION




In [ ]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nTarget Statistics:")
print(df["critical_temp"].describe())

In [ ]:
# ==========================================================
# BLOCK 4: FEATURE-TARGET SPLIT
# ==========================================================

X = df.drop("critical_temp", axis=1)
y = df["critical_temp"]

print("Feature Shape:", X.shape)
print("Target Shape :", y.shape)


BLOCK 5: TRAIN TEST SPLIT




In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Samples:", X_train.shape)
print("Testing Samples :", X_test.shape)

In [ ]:
# ==========================================================
# BLOCK 6: FEATURE SCALING
# ==========================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling Completed")

In [ ]:
# ==========================================================
# BLOCK 7: DEFINE GPR KERNEL
# ==========================================================

kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    *
    RBF(length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2))
    +
    WhiteKernel(noise_level=1)
)

print(kernel)

In [ ]:
# ==========================================================
# BLOCK 8: SUBSAMPLE TRAINING DATA
# ==========================================================

sample_size = 3000

X_train_gpr = X_train_scaled[:sample_size]
y_train_gpr = y_train.iloc[:sample_size]

print("Training on:", len(y_train_gpr), "samples")

In [ ]:
# ==========================================================
# BLOCK 9: TRAIN GPR MODEL
# ==========================================================

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    normalize_y=True,
    n_restarts_optimizer=5,
    random_state=42
)

gpr.fit(X_train_gpr, y_train_gpr)

print("Optimized Kernel:")
print(gpr.kernel_)

In [ ]:
# ==========================================================
# BLOCK 10: PREDICTIONS
# ==========================================================

y_pred, y_std = gpr.predict(
    X_test_scaled,
    return_std=True
)

print("Prediction Completed")

In [ ]:
# ==========================================================
# BLOCK 11: MODEL EVALUATION
# ==========================================================

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae,4))
print("MSE :", round(mse,4))
print("RMSE:", round(rmse,4))
print("R²  :", round(r2,4))

In [ ]:
# ==========================================================
# BLOCK 12: ACTUAL VS PREDICTED
# ==========================================================

plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.5
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.xlabel("Actual Critical Temperature")
plt.ylabel("Predicted Critical Temperature")

plt.title("Gaussian Process Regression")

plt.show()

In [ ]:
# ==========================================================
# BLOCK 13: UNCERTAINTY ANALYSIS
# ==========================================================

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Std_Deviation": y_std
})

results.head(10)

In [ ]:
# ==========================================================
# BLOCK 14: VISUALIZE UNCERTAINTY
# ==========================================================

subset = results.head(20)

plt.figure(figsize=(12,6))

plt.errorbar(
    range(len(subset)),
    subset["Predicted"],
    yerr=subset["Std_Deviation"],
    fmt='o'
)

plt.title("Prediction Uncertainty")
plt.ylabel("Critical Temperature")

plt.show()

In [ ]:
# ==========================================================
# BLOCK 15: SAVE RESULTS
# ==========================================================

results.to_csv(
    "GPR_Predictions.csv",
    index=False
)

print("Results saved successfully.")